In [15]:
import os
import pandas as pd
import numpy as np

os.chdir('C:/Users/lhcse/iCloudDrive/Learnings/Power-price-forecast')
df = pd.read_parquet('data/processed/merged_raw.parquet')

#3.1 fix NaNs

Since the german nuclear powerplant was shutdown and crossborders where treated as NaNs we fix that

In [16]:
 # Then check NaN counts and where they start/end
nan_cols = df.isnull().sum()
nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
print("NaN counts per column:")
print(nan_cols)

# For the worst offenders, check where NaNs are
for col in nan_cols.index[:5]:
    valid = df[col].dropna()
    print(f"\n{col}")
    print(f"  First valid: {valid.index.min()}")
    print(f"  Last valid:  {valid.index.max()}")
    print(f"  NaN count:   {df[col].isna().sum():,}")

NaN counts per column:
nuclear_gen_act     8075
net_export          6439
be_export           6431
be_import           6431
no_export           5879
no_import           5879
load_fc               24
residual_load_fc      24
fr_import              8
fr_export              8
dtype: int64

nuclear_gen_act
  First valid: 2020-01-01 00:00:00
  Last valid:  2024-01-30 11:00:00
  NaN count:   8,075

net_export
  First valid: 2020-09-25 00:00:00
  Last valid:  2024-12-31 23:00:00
  NaN count:   6,439

be_export
  First valid: 2020-09-25 00:00:00
  Last valid:  2024-12-31 23:00:00
  NaN count:   6,431

be_import
  First valid: 2020-09-25 00:00:00
  Last valid:  2024-12-31 23:00:00
  NaN count:   6,431

no_export
  First valid: 2020-09-02 00:00:00
  Last valid:  2024-12-31 23:00:00
  NaN count:   5,879


In [17]:
# ── Fix nuclear NaNs ───────────────────────────────────────────────────────
# Germany phased out nuclear April 2023 — SMARD reports as NaN after shutdown
# Correct value is 0
df['nuclear_gen_act'] = df['nuclear_gen_act'].fillna(0)

# ── Fix cross-border NaNs ──────────────────────────────────────────────────
# Cross-border data starts September 2020 in SMARD
# Fill early NaNs with 0 and forward-fill any gaps
cross_cols = [c for c in df.columns if any(
    x in c for x in ['export', 'import', 'net_export']
)]
df[cross_cols] = df[cross_cols].fillna(0)

# ── Fix small gaps in other columns ───────────────────────────────────────
# Forward-fill remaining small gaps (fr_import, fr_export, load_fc etc.)
df = df.ffill()

print("NaN counts after fix:")
print(df.isnull().sum()[df.isnull().sum() > 0])

NaN counts after fix:
Series([], dtype: int64)


#3.1 Forecasts error features

In [18]:
# ── Forecast-based features ────────────────────────────────────────────────
# These use day-ahead forecasts — available at prediction time (D-1)

df['ren_fc_mw']        = (df['wind_on_gen_fc'] + df['wind_off_gen_fc']
                          + df['solar_gen_fc'])
df['ren_load_ratio_fc'] = df['ren_fc_mw'] / df['load_fc']
df['ren_surplus_fc']    = (df['ren_fc_mw'] - df['load_fc']).clip(lower=0)

# ── Lagged forecast errors ─────────────────────────────────────────────────
# Compute errors first, then lag by 24h
# Hypothesis: yesterday's forecast error is informative about today's price

wind_on_error  = (df['wind_on_gen_act']  - df['wind_on_gen_fc'])  / df['load_act']
wind_off_error = (df['wind_off_gen_act'] - df['wind_off_gen_fc']) / df['load_act']
solar_error    = (df['solar_gen_act']    - df['solar_gen_fc'])    / df['load_act']
load_error     = (df['load_act']         - df['load_fc'])         / df['load_fc']
net_exp_norm   = df['net_export'] / df['load_act']

df['wind_on_error_lag24']  = wind_on_error.shift(24)
df['wind_off_error_lag24'] = wind_off_error.shift(24)
df['solar_error_lag24']    = solar_error.shift(24)
df['load_error_lag24']     = load_error.shift(24)
df['net_export_lag24']     = net_exp_norm.shift(24)

# ── Calendar features ──────────────────────────────────────────────────────
df['hour']       = df.index.hour
df['dayofweek']  = df.index.dayofweek
df['month']      = df.index.month
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
df['is_peak']    = df['hour'].between(8, 20).astype(int)

df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# ── Lagged price features ──────────────────────────────────────────────────
df['price_lag_24h']      = df['price_eur_mwh'].shift(24)
df['price_lag_48h']      = df['price_eur_mwh'].shift(48)
df['price_lag_168h']     = df['price_eur_mwh'].shift(168)

df['price_roll_7d_mean'] = df['price_eur_mwh'].shift(24).rolling(24*7).mean()
df['price_roll_7d_std']  = df['price_eur_mwh'].shift(24).rolling(24*7).std()

df = df.dropna()
df.to_parquet('data/processed/features.parquet')
print(f"Feature set: {df.shape[0]:,} rows, {df.shape[1]} columns")

Feature set: 43,652 rows, 70 columns
